In [ ]:
!pip install --user joblib scikit-learn pandas

In [2]:
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

print("============ STARTING MODEL TRAINING ============")

# 1. Load the noisy dataset with proper encoding standard
print("Loading 'processed_resume_data.csv'...")
df = pd.read_csv("processed_resume_data.csv", encoding="ISO-8859-1")
df['resume_text'] = df['resume_text'].fillna('')

X = df['resume_text']
y = df['label']

# 2. Split into Training (80%) and Testing (20%) datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Data split successful: Train size = {len(X_train)}, Test size = {len(X_test)}")

# 3. Create Scikit-Learn Classical Pipeline Architecture
print("Assembling Pipeline Architecture...")
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words='english', 
        max_features=4000, 
        ngram_range=(1, 2)
    )),
    ('classifier', RandomForestClassifier(
        n_estimators=100, 
        max_depth=15, 
        random_state=42,
        class_weight='balanced'
    ))
])

# 4. Train the Model
print("Training the Random Forest Fraud Classifier Model...")
model_pipeline.fit(X_train, y_train)

# 5. Realistic Model Evaluation Audit
print("\n================== MODEL PERFORMANCE EVALUATION ==================")
y_pred = model_pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred) * 100
print(f"Overall Model Predictive Accuracy: {accuracy:.2f}%")

print("\nDetailed Performance Class Metrics:")
print(classification_report(y_test, y_pred, target_names=['Genuine (0)', 'Suspicious (1)']))

print("Confusion Matrix Breakdown:")
cm = confusion_matrix(y_test, y_pred)
print(f"True Genuines Caught (TN): {cm[0][0]} | False Suspicious Flagged (FP): {cm[0][1]}")
print(f"False Genuines Leaked (FN): {cm[1][0]} | True Suspicious Caught (TP): {cm[1][1]}")

# 6. Serialized Export for API Integration
print("\n======================= EXPORT ARTIFACT =======================")
output_filename = "resume_fraud_model.pkl"
joblib.dump(model_pipeline, output_filename)
print(f"[SUCCESS] Saved production pipeline deployment file to: '{output_filename}'")
print("================================================================")

============ STARTING MODEL TRAINING ============
Loading 'processed_resume_data.csv'...
Data split successful: Train size = 1987, Test size = 497
Assembling Pipeline Architecture...
Training the Random Forest Fraud Classifier Model...

================== MODEL PERFORMANCE EVALUATION ==================
Overall Model Predictive Accuracy: 97.18%

Detailed Performance Class Metrics:
                precision    recall  f1-score   support

   Genuine (0)       0.97      1.00      0.98       390
Suspicious (1)       1.00      0.87      0.93       107

   avg / total       0.97      0.97      0.97       497

Confusion Matrix Breakdown:
True Genuines Caught (TN): 390 | False Suspicious Flagged (FP): 0
False Genuines Leaked (FN): 14 | True Suspicious Caught (TP): 93

======================= EXPORT ARTIFACT =======================
[SUCCESS] Saved production pipeline deployment file to: 'resume_fraud_model.pkl'
